In [ ]:
import os
from google.colab import userdata


# Définissez vos informations de manière sécurisée
GITHUB_USER = "koubra-gaby"
#GITHUB_TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

# Configuration de l'URL avec authentification
REPO_URL = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Cloner le dépôt si ce n'est pas déjà fait
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("Dépôt cloné avec succès !")
else:
    print("Le dépôt existe déjà.")


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 381 (delta 14), reused 9 (delta 3), pack-reused 348 (from 1)
Receiving objects: 100% (381/381), 22.53 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (178/178), done.
Dépôt cloné avec succès !


In [ ]:
# via python & Jupyter
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('update_shu'))

## training fastext model for chadian arabic corpus

In [ ]:
%%capture
!pip install gensim
!pip install nltk


In [ ]:
## Importing libraries
from gensim.models import FastText
from gensim.test.utils import common_texts

In [ ]:
import re
from nltk.tokenize import word_tokenize


def preparer_sentences_entrainer_fastext(fichier1, fichier2, file_to_save, lang):
    sentences = []
    if lang == "fr":
      patern = r"[^\w\s'-]"
    else :
      patern = r"[^\w\s']"
    for fichier in [fichier1, fichier2]:
        with open(fichier, "r", encoding="utf-8-sig") as f:
            for ligne in f:
                ligne = ligne.strip()

                if ligne:
                    # Remplacer les ponctuations par des espaces,
                    # sauf les apostrophes et les tirets
                    ligne = re.sub(
                        patern,
                        " ",
                        ligne,
                        flags=re.UNICODE
                    )

                    # Tokenisation mot par mot
                    tokens = word_tokenize(
                        ligne,
                        preserve_line=True
                    )

                    if tokens:
                        sentences.append(tokens)

      ## training fastext
    model = FastText(
      sentences,
      vector_size=1024,
      window=5,
      min_count=1,
      epochs=10,
      workers=4,
      sg=1,
  )
    model.wv.save_word2vec_format(file_to_save)

    print("Model trained and savec successfully")

    return model,sentences

In [ ]:
## Loading the corpus then tokenizing it
data_path = "/content/Chadian_arabic-update/Data/full_dataset/"
model_arabic,sentences_arabic = preparer_sentences_entrainer_fastext(data_path + "arabe_tchad3.txt",data_path + "arabic_not_bible.txt", data_path + "arabic_embedding.vec", lang = "shu")
sentences_arabic

In [ ]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_arabic.wv.most_similar(
    "malak",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

Words similar to 'ana':
amalak : 0.8625
malakt : 0.8166
malakti : 0.7944
halak : 0.7893
mursaalak : 0.7748
za'alak : 0.7733
seefak : 0.7626
malja'ak : 0.755
Za'alak : 0.7437
azalak : 0.7394


In [ ]:
model_fench,sentences_french = preparer_sentences_entrainer_fastext(data_path + "V_francais_3.txt",data_path + "french_not_bible.txt", data_path + "fench_embedding.vec", lang = "fr")
sentences_french

In [ ]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_fench.wv.most_similar(
    "bonjour",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

Words similar to 'ana':
Séjour : 0.8575
jour : 0.8081
lourde : 0.8006
journellement : 0.7679
lourd : 0.7671
bonheur : 0.7642
jour-ci : 0.7463
jour-aujourd'hui-en : 0.7273
lumignon : 0.7078
bonbon : 0.7045


In [ ]:
!git clone https://github.com/facebookresearch/MUSE.git

Cloning into 'MUSE'...
remote: Enumerating objects: 239, done.
remote: Total 239 (delta 0), reused 0 (delta 0), pack-reused 239 (from 1)
Receiving objects: 100% (239/239), 215.77 KiB | 30.82 MiB/s, done.
Resolving deltas: 100% (136/136), done.


In [ ]:
%cd /content/MUSE

from pathlib import Path

path = Path("src/utils.py")
code = path.read_text()

code = code.replace(
    "inspect.getargspec(optim_fn.__init__)[0]",
    "inspect.getfullargspec(optim_fn.__init__).args"
)

path.write_text(code)

print("Modification appliquée.")

/content/MUSE
Modification appliquée.


In [ ]:
!python unsupervised.py \
    --src_lang arb \
    --tgt_lang fr \
    --src_emb /content/Chadian_arabic-update/Data/full_dataset/fench_embedding.vec \
    --tgt_emb /content/Chadian_arabic-update/Data/full_dataset/arabic_embedding.vec \
    --emb_dim 1024 \
    --n_refinement 5 \
    --dis_most_frequent 20000,
    --dico_eval /content/arb_fr_eval.txt

# Projection vectorielle

In [ ]:
%%capture
!pip install -q gensim transformers scipy sentencepiece

### Charger les embedding NLLB et MUSE

In [ ]:
import os
import numpy as np
import torch

from gensim.models import KeyedVectors
from transformers import AutoTokenizer, AutoModel
from tokenizers import AddedToken

MODEL_NAME = "facebook/nllb-200-distilled-600M"

ARABIC_VEC_PATH = "/content/arabic_aligned.vec"
FRENCH_VEC_PATH = "/content/french_aligned.vec"
SUBWORDS_PATH = "/content/arabic_subwords.txt"

OUTPUT_DIR = "nllb_muse_aligned"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Charger les vecteurs MUSE
arabic_kv = KeyedVectors.load_word2vec_format(
    ARABIC_VEC_PATH,
    binary=False,
    encoding="utf-8"
)

french_kv = KeyedVectors.load_word2vec_format(
    FRENCH_VEC_PATH,
    binary=False,
    encoding="utf-8"
)

print("Vocabulaire arabe :", len(arabic_kv))
print("Vocabulaire français :", len(french_kv))
print("Dimension arabe :", arabic_kv.vector_size)
print("Dimension français :", french_kv.vector_size)

assert arabic_kv.vector_size == french_kv.vector_size == 1024

# Charger le tokenizer et NLLB
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="fra_Latn"
)

model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

nllb_embeddings = model.get_input_embeddings().weight

print("Dimension NLLB :", nllb_embeddings.shape[1])
assert nllb_embeddings.shape[1] == 1024

### contruire les ancres français (Vecteur français MUSE, vecteur français NLLB)

In [ ]:
def get_nllb_word_vector(word):
    """
    Représente un mot français par la moyenne des embeddings
    de ses sous-tokens NLLB.
    """
    encoded = tokenizer(
        word,
        add_special_tokens=False
    )["input_ids"]

    if not encoded:
        return None

    unk_id = tokenizer.unk_token_id

    # Exclure les mots qui deviennent uniquement des tokens inconnus.
    if unk_id is not None and all(i == unk_id for i in encoded):
        return None

    ids = torch.tensor(
        encoded,
        dtype=torch.long,
        device=nllb_embeddings.device
    )

    with torch.no_grad():
        vector = nllb_embeddings[ids].mean(dim=0)

    return vector.detach().cpu().float().numpy()


muse_vectors = []
nllb_vectors = []
anchor_words = []

# Intersection des vocabulaires
for word in french_kv.key_to_index:
    if word not in tokenizer.get_vocab():
        # Un mot peut être absent comme token unique tout en étant
        # représentable par plusieurs sous-tokens : on le garde.
        pass

    if not word.strip():
        continue

    muse_vec = np.asarray(french_kv[word], dtype=np.float32)
    nllb_vec = get_nllb_word_vector(word)

    if nllb_vec is None:
        continue

    if not np.isfinite(muse_vec).all():
        continue

    if not np.isfinite(nllb_vec).all():
        continue

    muse_vectors.append(muse_vec)
    nllb_vectors.append(nllb_vec)
    anchor_words.append(word)

X = np.stack(muse_vectors).astype(np.float64)
Y = np.stack(nllb_vectors).astype(np.float64)

print("Nombre d'ancres françaises :", len(anchor_words))
print("X :", X.shape)
print("Y :", Y.shape)

assert len(anchor_words) >= 1000, (
    "Trop peu d'ancres. Vérifie les fichiers MUSE et le tokenizer."
)

### Apprendre la transformation orthogonale

In [ ]:
from scipy.linalg import svd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

# Division train/test reproductible
indices = np.arange(len(X))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

X_train = X[train_idx]
Y_train = Y[train_idx]

# Moyennes calculées uniquement sur le train
mu_X = X_train.mean(axis=0)
mu_Y = Y_train.mean(axis=0)

# Centrage
X_train_c = X_train - mu_X
Y_train_c = Y_train - mu_Y

# Normalisation L2 ligne par ligne
X_train_n = normalize(X_train_c, norm="l2")
Y_train_n = normalize(Y_train_c, norm="l2")

# Procrustes : trouver R tel que X_train_n @ R ≈ Y_train_n
M = X_train_n.T @ Y_train_n

U, S, Vt = svd(M, full_matrices=False)
R = U @ Vt

print("Transformation R :", R.shape)
print("Erreur d'orthogonalité :",
      np.linalg.norm(R.T @ R - np.eye(R.shape[0])))

#### Evaluer sur les ancres reservés

In [ ]:
X_test = X[test_idx]
Y_test = Y[test_idx]

X_test_c = normalize(X_test - mu_X, norm="l2")
Y_test_c = normalize(Y_test - mu_Y, norm="l2")

Y_pred = X_test_c @ R

cosines = np.sum(Y_pred * Y_test_c, axis=1)

print("Ancres de test :", len(test_idx))
print("Similarité cosinus moyenne :", float(cosines.mean()))
print("Médiane :", float(np.median(cosines)))
print("10e percentile :", float(np.percentile(cosines, 10)))

### Definir la projection des vectuers MUSE

In [ ]:
def project_muse_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)

    centered = vector - mu_X
    norm = np.linalg.norm(centered)

    if norm < 1e-12:
        raise ValueError("Vecteur MUSE quasi nul après centrage.")

    centered = centered / norm
    projected = centered @ R

    # Revenir à l'espace cible centré
    projected = projected + mu_Y

    return projected.astype(np.float32)

### Construire un vecteur pour chaque sous-mot arabe

In [ ]:
with open(SUBWORDS_PATH, "r", encoding="utf-8") as f:
    subwords = list(dict.fromkeys(
        line.strip() for line in f if line.strip()
    ))

print("Sous-mots candidats :", len(subwords))


def build_subword_muse_vector(subword, kv, min_matches=3):
    """
    Moyenne les vecteurs des mots MUSE contenant le sous-mot.
    min_matches limite les estimations reposant sur trop peu de mots.
    """
    matches = [
        word for word in kv.key_to_index
        if subword in word
    ]

    if len(matches) < min_matches:
        return None

    vectors = np.stack([kv[word] for word in matches]).astype(np.float32)

    return vectors.mean(axis=0)


subword_vectors = {}
not_found = []

for subword in subwords:
    vector = build_subword_muse_vector(
        subword,
        arabic_kv,
        min_matches=3
    )

    if vector is None:
        not_found.append(subword)
        continue

    subword_vectors[subword] = vector

print("Sous-mots représentés :", len(subword_vectors))
print("Sous-mots ignorés :", len(not_found))
print("Exemples ignorés :", not_found[:20])

### Projeter les sous-mots et ajouter les tokens à NLLB

In [ ]:
from collections import OrderedDict

# Ne conserver que les nouveaux tokens
existing_vocab = tokenizer.get_vocab()

tokens_to_add = [
    subword for subword in subword_vectors
    if subword not in existing_vocab
]

print("Nouveaux tokens à ajouter :", len(tokens_to_add))

# Projeter les vecteurs avant l'ajout
projected_vectors = {
    subword: project_muse_vector(subword_vectors[subword])
    for subword in tokens_to_add
}

# Ajouter les tokens au tokenizer
added_tokens = [
    AddedToken(token, normalized=False, special=False)
    for token in tokens_to_add
]

num_added = tokenizer.add_tokens(added_tokens)

print("Tokens effectivement ajoutés :", num_added)

# Important : redimensionner la matrice du modèle
model.resize_token_embeddings(len(tokenizer))

embedding_layer = model.get_input_embeddings()
embedding_dim = embedding_layer.weight.shape[1]

assert embedding_dim == 1024

# Initialiser les lignes des nouveaux tokens
initialized = 0
skipped = []

with torch.no_grad():
    for subword in tokens_to_add:
        token_id = tokenizer.convert_tokens_to_ids(subword)

        if token_id is None:
            skipped.append(subword)
            continue

        vector = projected_vectors[subword]

        if vector.shape != (embedding_dim,):
            skipped.append(subword)
            continue

        embedding_layer.weight[token_id].copy_(
            torch.tensor(
                vector,
                dtype=embedding_layer.weight.dtype,
                device=embedding_layer.weight.device
            )
        )

        initialized += 1

print("Tokens initialisés :", initialized)
print("Tokens ignorés :", len(skipped))

### Sauvegarder le modèle, le tokenizer et la transformation

In [ ]:
import json

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sauvegarder les paramètres du modèle et le tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Sauvegarder la transformation et les moyennes
np.savez(
    os.path.join(OUTPUT_DIR, "muse_to_nllb_procrustes.npz"),
    R=R,
    mu_X=mu_X,
    mu_Y=mu_Y
)

metadata = {
    "base_model": MODEL_NAME,
    "embedding_dimension": int(embedding_dim),
    "number_of_french_anchors": int(len(anchor_words)),
    "number_of_train_anchors": int(len(train_idx)),
    "number_of_test_anchors": int(len(test_idx)),
    "number_of_new_tokens_added": int(num_added),
    "number_of_tokens_initialized": int(initialized),
    "mean_test_cosine": float(cosines.mean()),
    "subword_initialization": "mean of MUSE word vectors containing the subword"
}

with open(
    os.path.join(OUTPUT_DIR, "alignment_metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Modèle et transformation sauvegardés dans :", OUTPUT_DIR)